In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import networkx as nx
import itertools
import random
import math
random.seed(2023)
import numpy as np
np.random.seed(2023)
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
colors = ['#104680','#317CB7','#6DADD1','#B6D7E8','#E9F1F4','#FBE3D5','#F6B293','#DC6D57','#B72230','#6D011F']
cmap = ListedColormap(colors)
%matplotlib inline

In [3]:
file_path = "../result/pmfg_directed.edgelist"
G = nx.read_edgelist(file_path, create_using=nx.DiGraph(), nodetype=str)

In [4]:
import numpy as np
import pandas as pd
import networkx as nx


# 1. 单向风险溢出指数
def compute_single_risk_outflow(G):
    """计算单向风险溢出指数 S_{i -> j}^H"""
    # 转换为加权邻接矩阵
    adj_matrix = nx.to_numpy_array(G, weight='weight')
    return 100 * adj_matrix

# 2. 传染性指数 OUT_i^H
def compute_outflow_index(G):
    """计算传染性指数 OUT_i^H"""
    outflow_index = {}
    for node in G.nodes():
        outflow_index[node] = sum(data['weight'] for _, _, data in G.out_edges(node, data=True))
    return outflow_index

# 3. 脆弱性指数 IN_i^H
def compute_inflow_index(G):
    """计算脆弱性指数 IN_i^H"""
    inflow_index = {}
    for node in G.nodes():
        inflow_index[node] = sum(data['weight'] for _, _, data in G.in_edges(node, data=True))
    return inflow_index

# 4. 净溢出指数 NET_i^H
def compute_net_outflow_index(outflow_index, inflow_index):
    """计算净溢出指数 NET_i^H"""
    net_outflow_index = {}
    for node in outflow_index.keys():
        net_outflow_index[node] = outflow_index[node] - inflow_index[node]
    return net_outflow_index

# 5. 总体关联性指数 TOTAL^H
def compute_total_connectivity(G):
    """计算总体关联性指数 TOTAL^H"""
    total_weight = sum(data['weight'] for _, _, data in G.edges(data=True))
    num_nodes = G.number_of_nodes()
    return total_weight / num_nodes

# 计算各项指数
single_risk_outflow = compute_single_risk_outflow(G)  # 单向风险溢出指数矩阵
outflow_index = compute_outflow_index(G)             # 传染性指数
inflow_index = compute_inflow_index(G)               # 脆弱性指数
net_outflow_index = compute_net_outflow_index(outflow_index, inflow_index)  # 净溢出指数
total_connectivity = compute_total_connectivity(G)   # 总体关联性指数

# 将结果输出为 DataFrame 便于查看
results = pd.DataFrame({
    "机构": list(G.nodes()),
    "传染性指数 (OUT_i^H)": outflow_index.values(),
    "脆弱性指数 (IN_i^H)": inflow_index.values(),
    "净溢出指数 (NET_i^H)": net_outflow_index.values()
})

# 输出计算结果
print("单向风险溢出指数矩阵:")
print(pd.DataFrame(single_risk_outflow, columns=list(G.nodes()), index=list(G.nodes())))
print("\n风险溢出指数汇总:")
print(results)
print(f"\n总体关联性指数 (TOTAL^H): {total_connectivity:.4f}")

单向风险溢出指数矩阵:
        CMB   BOCOM    PAB   CMBC   SPDB    CSC  CITICS    SXZ    ECP    GIC  \
CMB     0.0   763.0  526.0  508.0  503.0    0.0     0.0    0.0    0.0    0.0   
BOCOM   0.0     0.0    0.0    0.0    0.0    0.0     0.0    0.0    0.0    0.0   
PAB     0.0   492.0    0.0    0.0    0.0    0.0     0.0    0.0    0.0    0.0   
CMBC    0.0   758.0  516.0    0.0  590.0    0.0     0.0    0.0    0.0    0.0   
SPDB    0.0   605.0    0.0    0.0    0.0    0.0     0.0    0.0    0.0    0.0   
CSC     0.0     0.0    0.0    0.0    0.0    0.0   823.0  668.0    0.0    0.0   
CITICS  0.0     0.0    0.0    0.0    0.0    0.0     0.0  670.0    0.0    0.0   
SXZ     0.0     0.0    0.0    0.0    0.0    0.0     0.0    0.0    0.0    0.0   
ECP     0.0     0.0    0.0    0.0    0.0    0.0     0.0  224.0    0.0  116.0   
GIC     0.0     0.0    0.0    0.0    0.0    0.0   485.0  537.0    0.0    0.0   
DFC     0.0     0.0    0.0    0.0    0.0    0.0   319.0    0.0    0.0    0.0   
XYR     0.0     0.0    0.0  

In [5]:
import pandas as pd
import numpy as np

single_risk_outflow_df = pd.DataFrame(single_risk_outflow)
output_file = "../result/静态网络风险指数-pmfg/单向风险溢出矩阵.xlsx"

single_risk_outflow_df.to_excel(output_file, index=False)
print(f"文件已保存为: {output_file}")

文件已保存为: ../result/静态网络风险指数-pmfg/单向风险溢出矩阵.xlsx


In [6]:
import os

# 保存单个指数到 Excel
def save_index_to_excel(data, index_name, file_name):
    """
    保存某个风险指数到单独的 Excel 文件
    :param data: 数据 (dict 或矩阵)
    :param index_name: 指数名称
    :param file_name: 文件名
    """
    if isinstance(data, dict):  # 如果是字典
        df = pd.DataFrame({
            "机构": list(data.keys()),
            index_name: list(data.values())
        })
    else:  # 如果是矩阵
        df = pd.DataFrame(
            data,
            columns=list(G.nodes()),
            index=list(G.nodes())
        )
    
    df.to_excel(file_name, index=True if isinstance(data, np.ndarray) else False)
    print(f"{index_name} 已保存为 {os.path.abspath(file_name)}")


# 分别保存四个风险指数到单独的 Excel 文件
save_index_to_excel(
    single_risk_outflow, 
    "单向风险溢出指数", 
    "../result/单向风险溢出指数.xlsx"
)
save_index_to_excel(
    outflow_index, 
    "传染性指数 (OUT_i^H)", 
    "../result/传染性指数.xlsx"
)
save_index_to_excel(
    inflow_index, 
    "脆弱性指数 (IN_i^H)", 
    "../result/脆弱性指数.xlsx"
)
save_index_to_excel(
    net_outflow_index, 
    "净溢出指数 (NET_i^H)", 
    "../result/净溢出指数.xlsx"
)

单向风险溢出指数 已保存为 /Users/liujiyu/Documents/RUC/2025课程学习/2025-金融理论与政策-马勇/刘及雨-中国金融机构风险溢出网络与风险传染探究/result/单向风险溢出指数.xlsx
传染性指数 (OUT_i^H) 已保存为 /Users/liujiyu/Documents/RUC/2025课程学习/2025-金融理论与政策-马勇/刘及雨-中国金融机构风险溢出网络与风险传染探究/result/传染性指数.xlsx
脆弱性指数 (IN_i^H) 已保存为 /Users/liujiyu/Documents/RUC/2025课程学习/2025-金融理论与政策-马勇/刘及雨-中国金融机构风险溢出网络与风险传染探究/result/脆弱性指数.xlsx
净溢出指数 (NET_i^H) 已保存为 /Users/liujiyu/Documents/RUC/2025课程学习/2025-金融理论与政策-马勇/刘及雨-中国金融机构风险溢出网络与风险传染探究/result/净溢出指数.xlsx
